# Introduction to SQL in Python (SQLite)

Welcome! This notebook is a beginner-friendly guide to using **SQL** inside Python using the built-in **`sqlite3`** module.

**Topics Covered:**
1. **Introduction to SQL in Python** - what SQL is, why SQLite, connecting to a database
2. **Designing and changing** databases and tables (`CREATE`, `ALTER`, `DROP`)
3. **CRUD operations** - `INSERT`, `SELECT`, `UPDATE`, `DELETE`
4. **Filtering data** with the `WHERE` clause (`AND`, `OR`, `LIKE`, `IN`, `BETWEEN`, `ORDER BY`, `LIMIT`)
5. **Practice Exercises** - hands-on mini tasks

---

**How to use this notebook:** Click on a code cell and press `Shift + Enter` to run it.

> No installation is required - `sqlite3` comes bundled with Python.

# 1. Introduction to SQL in Python

## 1.1 What is SQL?

**SQL** (Structured Query Language) is the standard language used to talk to **relational databases**. A relational database stores data in **tables** (like spreadsheets) with rows and columns.

With SQL you can:
- **Create** databases and tables
- **Insert** new records
- **Read (query)** existing records
- **Update** or **delete** records
- **Filter**, **sort**, and **join** data

## 1.2 Why SQLite?

- **Zero setup** - no server to install, the whole database is a single `.db` file
- **Built into Python** - just `import sqlite3`
- Perfect for **learning**, prototypes, and small apps

## 1.3 The workflow in Python

```
connect  ->  cursor  ->  execute SQL  ->  fetch results  ->  commit  ->  close
```

| Step | Purpose |
|------|---------|
| `connect()` | Open (or create) the database file |
| `cursor()` | Object used to run SQL statements |
| `execute()` | Run a single SQL statement |
| `fetchall()` / `fetchone()` | Read the results of a `SELECT` |
| `commit()` | Save the changes to disk |
| `close()` | Close the connection |

In [ ]:
# Import the built-in sqlite3 module
import sqlite3

# Create (or open) a database file called school.db
# If the file does not exist, sqlite3 will create it automatically.
conn = sqlite3.connect("school.db")

# A cursor is what we use to actually run SQL statements
cursor = conn.cursor()

print("Connected to SQLite version :", sqlite3.sqlite_version)
print("Connection object           :", conn)
print("Cursor object               :", cursor)

In [ ]:
# A quick 'Hello World' SQL query - ask SQLite for its version
cursor.execute("SELECT sqlite_version();")

# fetchone() returns the first row of the result
row = cursor.fetchone()
print("SQLite says :", row)

---

# 2. Designing and Changing Databases & Tables

## 2.1 Common Data Types in SQLite

| Type | Stores |
|------|--------|
| `INTEGER` | Whole numbers |
| `REAL` | Decimal numbers |
| `TEXT` | Strings |
| `BLOB` | Binary data (images, files) |
| `NULL` | No value |

## 2.2 Constraints

Rules that keep our data clean.

| Constraint | Meaning |
|------------|---------|
| `PRIMARY KEY` | Unique identifier for each row |
| `AUTOINCREMENT` | SQLite gives the next number automatically |
| `NOT NULL` | Column must have a value |
| `UNIQUE` | No two rows can share the same value |
| `DEFAULT` | Value used when none is supplied |
| `CHECK` | Custom condition the value must satisfy |

## 2.3 Key SQL Statements

| Statement | Purpose |
|-----------|---------|
| `CREATE TABLE` | Make a new table |
| `ALTER TABLE` | Add / rename a column, rename the table |
| `DROP TABLE` | Remove a table completely |

In [ ]:
# Start clean - if the table already exists from an earlier run, drop it.
cursor.execute("DROP TABLE IF EXISTS students;")

# CREATE TABLE - define columns, types and constraints
create_sql = """
CREATE TABLE students (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    name    TEXT    NOT NULL,
    age     INTEGER CHECK (age > 0),
    grade   TEXT    DEFAULT 'N/A',
    email   TEXT    UNIQUE
);
"""

cursor.execute(create_sql)
conn.commit()          # always commit after changing the database

print("Table 'students' created successfully.")

In [ ]:
# Look at the structure of the table we just created
cursor.execute("PRAGMA table_info(students);")

print(f"{'cid':<4}{'name':<10}{'type':<10}{'notnull':<10}{'default':<10}{'pk'}")
print("-" * 55)
for col in cursor.fetchall():
    cid, name, ctype, notnull, dflt, pk = col
    print(f"{cid:<4}{name:<10}{ctype:<10}{notnull:<10}{str(dflt):<10}{pk}")

In [ ]:
# ALTER TABLE - add a new column
cursor.execute("ALTER TABLE students ADD COLUMN city TEXT DEFAULT 'Unknown';")
conn.commit()

# Check the table structure again
cursor.execute("PRAGMA table_info(students);")
for col in cursor.fetchall():
    print(col)

In [ ]:
# ALTER TABLE - rename a column (SQLite 3.25+)
cursor.execute("ALTER TABLE students RENAME COLUMN city TO hometown;")
conn.commit()

cursor.execute("PRAGMA table_info(students);")
for col in cursor.fetchall():
    print(col)

In [ ]:
# DROP TABLE - remove a table completely
# Let's create a throwaway table and then drop it.
cursor.execute("CREATE TABLE temp_data (id INTEGER, note TEXT);")
print("Created 'temp_data'.")

cursor.execute("DROP TABLE temp_data;")
print("Dropped 'temp_data'.")

conn.commit()

# List all tables that still exist in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Current tables :", cursor.fetchall())

---

# 3. CRUD Operations

**CRUD** stands for the four basic actions on data:

| Letter | SQL keyword | Meaning |
|--------|-------------|---------|
| **C**reate | `INSERT` | Add new rows |
| **R**ead | `SELECT` | Fetch rows |
| **U**pdate | `UPDATE` | Change existing rows |
| **D**elete | `DELETE` | Remove rows |

## 3.1 Safe queries with `?` placeholders

Never build SQL by string-concatenating user input - that leads to **SQL injection**.
Instead, use `?` placeholders and pass values as a tuple:

```python
cursor.execute("INSERT INTO students(name, age) VALUES (?, ?);", ("Alice", 20))
```

## 3.2 CREATE - `INSERT INTO`

In [ ]:
# Insert a single row - notice we skip 'id' because it is AUTOINCREMENT
cursor.execute(
    "INSERT INTO students (name, age, grade, email, hometown) VALUES (?, ?, ?, ?, ?);",
    ("Alice", 20, "A", "alice@example.com", "Mumbai")
)
conn.commit()

print("Inserted 1 row. Row id =", cursor.lastrowid)

In [ ]:
# Insert MANY rows at once with executemany() - much faster than a loop
many_students = [
    ("Bob",     22, "B", "bob@example.com",     "Delhi"),
    ("Charlie", 19, "A", "charlie@example.com", "Pune"),
    ("Diana",   21, "C", "diana@example.com",   "Mumbai"),
    ("Ethan",   23, "B", "ethan@example.com",   "Bangalore"),
    ("Fiona",   20, "A", "fiona@example.com",   "Delhi"),
]

cursor.executemany(
    "INSERT INTO students (name, age, grade, email, hometown) VALUES (?, ?, ?, ?, ?);",
    many_students
)
conn.commit()

print("Rows inserted this run :", cursor.rowcount)

## 3.3 READ - `SELECT`

| Method | Returns |
|--------|---------|
| `fetchone()` | The next row (tuple) or `None` |
| `fetchmany(n)` | A list with the next `n` rows |
| `fetchall()` | A list with **all** remaining rows |

In [ ]:
# SELECT * - read ALL columns of ALL rows
cursor.execute("SELECT * FROM students;")

rows = cursor.fetchall()
print(f"Total rows: {len(rows)}\n")

for row in rows:
    print(row)

In [ ]:
# SELECT only the columns you need
cursor.execute("SELECT name, grade FROM students;")

for name, grade in cursor.fetchall():
    print(f"{name:<10} -> Grade {grade}")

In [ ]:
# fetchone() - get just the first matching row
cursor.execute("SELECT * FROM students;")

first = cursor.fetchone()
print("First row :", first)

# fetchmany(n) - get the next n rows
next_two = cursor.fetchmany(2)
print("Next two  :", next_two)

In [ ]:
# Reading rows as dictionaries by setting a row_factory
conn.row_factory = sqlite3.Row       # each row acts like a dict
cursor = conn.cursor()

cursor.execute("SELECT id, name, age FROM students;")
for row in cursor.fetchall():
    # access by column name instead of index
    print(f"ID={row['id']} | Name={row['name']} | Age={row['age']}")

# Reset back to default so the rest of the notebook stays simple
conn.row_factory = None
cursor = conn.cursor()

## 3.4 UPDATE

**Always** include a `WHERE` clause on `UPDATE` - without it you will change **every** row!

In [ ]:
# Update ONE row - change Bob's grade
cursor.execute(
    "UPDATE students SET grade = ? WHERE name = ?;",
    ("A", "Bob")
)
conn.commit()

print("Rows updated :", cursor.rowcount)

# Confirm the change
cursor.execute("SELECT name, grade FROM students WHERE name = 'Bob';")
print(cursor.fetchone())

In [ ]:
# Update MULTIPLE rows - promote every student in Mumbai to grade 'A'
cursor.execute(
    "UPDATE students SET grade = ? WHERE hometown = ?;",
    ("A", "Mumbai")
)
conn.commit()

print("Rows updated :", cursor.rowcount)

cursor.execute("SELECT name, hometown, grade FROM students;")
for row in cursor.fetchall():
    print(row)

## 3.5 DELETE

Just like `UPDATE`, **always** include a `WHERE` clause on `DELETE` unless you really want to empty the table.

In [ ]:
# Delete a specific row
cursor.execute("DELETE FROM students WHERE name = ?;", ("Ethan",))
conn.commit()

print("Rows deleted :", cursor.rowcount)

# Show remaining rows
cursor.execute("SELECT id, name FROM students;")
for row in cursor.fetchall():
    print(row)

In [ ]:
# Delete everything from a table (structure stays intact)
# NOTE: commented out so we don't wipe our practice data.
# cursor.execute("DELETE FROM students;")
# conn.commit()

# Count the remaining rows
cursor.execute("SELECT COUNT(*) FROM students;")
print("Students in table :", cursor.fetchone()[0])

---

# 4. Filtering Data with `WHERE`

The `WHERE` clause lets you pick **only the rows that match a condition**.

```sql
SELECT columns FROM table WHERE condition;
```

## 4.1 Comparison Operators

| Operator | Meaning |
|----------|---------|
| `=` | Equal to |
| `!=` or `<>` | Not equal |
| `<` `>` `<=` `>=` | Less / greater / etc. |
| `BETWEEN a AND b` | In a range (inclusive) |
| `IN (v1, v2, ...)` | Matches any value in the list |
| `LIKE 'pattern'` | Pattern match (`%` = any chars, `_` = one char) |
| `IS NULL` / `IS NOT NULL` | Missing value check |

## 4.2 Logical Operators

| Operator | Purpose |
|----------|---------|
| `AND` | Both conditions must be true |
| `OR` | At least one condition true |
| `NOT` | Reverses a condition |

## 4.3 Helpful Extras

| Clause | Purpose |
|--------|---------|
| `ORDER BY col [ASC|DESC]` | Sort the results |
| `LIMIT n` | Return only `n` rows |
| `DISTINCT` | Remove duplicate values |

In [ ]:
# Before we filter, let's re-seed a fresh copy of the data so results are consistent.
cursor.execute("DELETE FROM students;")

sample = [
    ("Alice",   20, "A", "alice@example.com",   "Mumbai"),
    ("Bob",     22, "B", "bob@example.com",     "Delhi"),
    ("Charlie", 19, "A", "charlie@example.com", "Pune"),
    ("Diana",   21, "C", "diana@example.com",   "Mumbai"),
    ("Ethan",   23, "B", "ethan@example.com",   "Bangalore"),
    ("Fiona",   20, "A", "fiona@example.com",   "Delhi"),
    ("George",  25, "C", "george@example.com",  "Pune"),
    ("Hannah",  18, "A", "hannah@example.com",  "Mumbai"),
]

cursor.executemany(
    "INSERT INTO students (name, age, grade, email, hometown) VALUES (?, ?, ?, ?, ?);",
    sample
)
conn.commit()
print("Fresh data loaded. Total rows :", cursor.execute("SELECT COUNT(*) FROM students;").fetchone()[0])

In [ ]:
# Simple WHERE - equality
cursor.execute("SELECT name, grade FROM students WHERE grade = 'A';")

print("Grade A students:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# Comparison operators
cursor.execute("SELECT name, age FROM students WHERE age > 20;")

print("Older than 20:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# Combining with AND / OR
cursor.execute("SELECT name, age, grade FROM students WHERE grade = 'A' AND age >= 20;")
print("Grade A AND age >= 20:")
for row in cursor.fetchall():
    print(" ", row)

print()

cursor.execute("SELECT name, hometown FROM students WHERE hometown = 'Mumbai' OR hometown = 'Delhi';")
print("From Mumbai OR Delhi:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# BETWEEN - inclusive range
cursor.execute("SELECT name, age FROM students WHERE age BETWEEN 19 AND 21;")

print("Age between 19 and 21:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# IN - match against a list of values
cursor.execute("SELECT name, hometown FROM students WHERE hometown IN ('Mumbai', 'Pune');")

print("From Mumbai or Pune:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# LIKE - pattern matching
#  %  -> any number of characters (including zero)
#  _  -> exactly one character

# Names starting with 'A'
cursor.execute("SELECT name FROM students WHERE name LIKE 'A%';")
print("Starts with A :", cursor.fetchall())

# Names ending with 'a'
cursor.execute("SELECT name FROM students WHERE name LIKE '%a';")
print("Ends with a   :", cursor.fetchall())

# Names that contain 'an'
cursor.execute("SELECT name FROM students WHERE name LIKE '%an%';")
print("Contains 'an' :", cursor.fetchall())

# 4-letter names
cursor.execute("SELECT name FROM students WHERE name LIKE '____';")
print("4 letters     :", cursor.fetchall())

In [ ]:
# NOT - reverse a condition
cursor.execute("SELECT name, grade FROM students WHERE NOT grade = 'A';")

print("Not grade A:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# ORDER BY - sort the results
cursor.execute("SELECT name, age FROM students ORDER BY age ASC;")
print("By age (youngest first):")
for row in cursor.fetchall():
    print(" ", row)

print()

cursor.execute("SELECT name, age FROM students ORDER BY age DESC;")
print("By age (oldest first):")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# LIMIT - only return the first N rows
cursor.execute("SELECT name, age FROM students ORDER BY age DESC LIMIT 3;")

print("Top 3 oldest students:")
for row in cursor.fetchall():
    print(" ", row)

In [ ]:
# DISTINCT - remove duplicate values
cursor.execute("SELECT DISTINCT hometown FROM students;")

print("Unique hometowns:")
for row in cursor.fetchall():
    print(" ", row[0])

In [ ]:
# Aggregate functions with WHERE
# COUNT, AVG, SUM, MIN, MAX
cursor.execute("SELECT COUNT(*), AVG(age), MIN(age), MAX(age) FROM students WHERE grade = 'A';")
count, avg, mn, mx = cursor.fetchone()

print("Grade A stats")
print(f"  Count : {count}")
print(f"  Avg   : {avg:.2f}")
print(f"  Min   : {mn}")
print(f"  Max   : {mx}")

In [ ]:
# GROUP BY - group rows that share a value, and use an aggregate for each group
cursor.execute("SELECT hometown, COUNT(*) FROM students GROUP BY hometown ORDER BY COUNT(*) DESC;")

print("Students per hometown:")
for town, count in cursor.fetchall():
    print(f"  {town:<10} : {count}")

---

# 5. Practice Exercises

Time to try it yourself! Below is a mini project - a small **`books`** database.

**Tasks:**
1. Create a `books` table (id, title, author, year, price, genre).
2. Insert at least 8 sample books.
3. Select all books written after the year 2000.
4. Find the average price of each genre.
5. Update the price of one book.
6. Delete a book by its title.
7. List all unique genres.
8. Find the top 3 most expensive books.

Try each one on your own first, then check the solution cell.

## Exercise 1: Create the `books` table

In [ ]:
cursor.execute("DROP TABLE IF EXISTS books;")

cursor.execute("""
CREATE TABLE books (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    year    INTEGER,
    price   REAL    CHECK (price >= 0),
    genre   TEXT
);
""")
conn.commit()

# Verify structure
cursor.execute("PRAGMA table_info(books);")
for col in cursor.fetchall():
    print(col)

## Exercise 2: Insert sample books

In [ ]:
books = [
    ("The Alchemist",         "Paulo Coelho",       1988, 299.0, "Fiction"),
    ("Atomic Habits",         "James Clear",        2018, 499.0, "Self-Help"),
    ("Sapiens",               "Yuval Noah Harari",  2011, 599.0, "History"),
    ("Clean Code",            "Robert C. Martin",   2008, 899.0, "Technology"),
    ("Rich Dad Poor Dad",     "Robert Kiyosaki",    1997, 350.0, "Finance"),
    ("Deep Work",             "Cal Newport",        2016, 450.0, "Self-Help"),
    ("The Pragmatic Programmer", "Andy Hunt",       1999, 950.0, "Technology"),
    ("Educated",              "Tara Westover",      2018, 425.0, "Memoir"),
]

cursor.executemany(
    "INSERT INTO books (title, author, year, price, genre) VALUES (?, ?, ?, ?, ?);",
    books
)
conn.commit()

print("Inserted rows :", cursor.rowcount)

## Exercise 3: Books written after the year 2000

In [ ]:
cursor.execute("SELECT title, year FROM books WHERE year > 2000 ORDER BY year;")

for title, year in cursor.fetchall():
    print(f"{year}  |  {title}")

## Exercise 4: Average price per genre

In [ ]:
cursor.execute("""
SELECT genre, ROUND(AVG(price), 2) AS avg_price
FROM books
GROUP BY genre
ORDER BY avg_price DESC;
""")

for genre, avg_price in cursor.fetchall():
    print(f"{genre:<12} -> {avg_price}")

## Exercise 5: Update the price of a book

In [ ]:
cursor.execute("UPDATE books SET price = ? WHERE title = ?;", (349.0, "The Alchemist"))
conn.commit()

cursor.execute("SELECT title, price FROM books WHERE title = 'The Alchemist';")
print("After update :", cursor.fetchone())

## Exercise 6: Delete a book by title

In [ ]:
cursor.execute("DELETE FROM books WHERE title = ?;", ("Rich Dad Poor Dad",))
conn.commit()

print("Rows deleted :", cursor.rowcount)

cursor.execute("SELECT title FROM books;")
print("Remaining books:")
for row in cursor.fetchall():
    print(" ", row[0])

## Exercise 7: List unique genres

In [ ]:
cursor.execute("SELECT DISTINCT genre FROM books ORDER BY genre;")

print("Unique genres:")
for row in cursor.fetchall():
    print(" ", row[0])

## Exercise 8: Top 3 most expensive books

In [ ]:
cursor.execute("SELECT title, price FROM books ORDER BY price DESC LIMIT 3;")

print("Top 3 most expensive:")
for title, price in cursor.fetchall():
    print(f"  {title:<28} Rs.{price}")

---

# 6. Always Close the Connection

When you finish using a database, close the connection so the file is released and any pending changes are flushed.

In [ ]:
conn.commit()
conn.close()
print("Connection closed.")

## Bonus tip - use `with` for automatic commit/close

The `with` statement automatically **commits** on success and **rolls back** on error. You still need to close the connection, but it is a very clean pattern.

In [ ]:
import sqlite3

with sqlite3.connect("school.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM students;")
    print("Students in DB :", cursor.fetchone()[0])

    cursor.execute("SELECT COUNT(*) FROM books;")
    print("Books in DB    :", cursor.fetchone()[0])

conn.close()
print("Done.")

---

## Quick Summary

| Concept | One-line takeaway |
|---------|-------------------|
| `sqlite3.connect(file)` | Opens (or creates) a database file |
| `cursor.execute(sql, params)` | Runs a single SQL statement safely |
| `executemany(sql, list)` | Runs the same statement over many rows |
| `fetchone / fetchmany / fetchall` | Read query results |
| `commit()` | Save changes to disk |
| `CREATE TABLE` | Define a new table + constraints |
| `ALTER TABLE` | Add / rename columns |
| `DROP TABLE` | Delete a table entirely |
| `INSERT / SELECT / UPDATE / DELETE` | The CRUD verbs |
| `WHERE` | Filter rows by a condition |
| `AND / OR / NOT` | Combine conditions |
| `BETWEEN / IN / LIKE` | Extra WHERE helpers |
| `ORDER BY`, `LIMIT`, `DISTINCT` | Sort, cut off, deduplicate |
| `GROUP BY` + `COUNT/AVG/SUM/MIN/MAX` | Aggregate by group |
| `?` placeholders | Prevent SQL injection |

**Nice work!** You now know enough SQL to manage a small database with Python. Try changing the sample data, add new columns, or invent your own tables (movies, songs, todos) and repeat the exercises.